# Expectimax

_Artificial Intelligence (CS221)_

**When the opponent is random, not perfect, average the outcomes instead of taking the worst.**

Minimax assumes a perfect opponent. But sometimes the opponent is random, or there is a dice roll.

     Expectimax replaces the opponent's min with an average.

---

This notebook builds **Expectimax** from the ground up, one small step at a time. We'll see how a single recursive function captures *both* minimax and expectimax, and how swapping "take the worst" for "take the average" changes the move you'd pick. Run each cell, read the note above it, then visualize how the two opponent models compare. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

A game tree alternates between **our** moves and the **opponent's** moves. We build the evaluator in three steps: (1) lay out a tiny game tree, (2) write one recursive function that handles maximizing, minimizing, and chance nodes, (3) compare the minimax value (worst-case opponent) against the expectimax value (random opponent).

### Step 1 — Lay out a tiny game tree

A game tree is nested lists. The outer list is **our turn** (we'll maximize over it). Each inner list is the **opponent's turn** — a set of replies they could make. The plain numbers at the bottom are **leaf utilities**: the final scores we'd receive.

Here we have two moves available to us, and the opponent has three responses to each.

In [ ]:
# Two subtrees, one per move we could make.
# Each subtree holds the opponent's three possible replies (leaf utilities).
tree = [[3, 12, 8], [4, 6, 2]]

print("our moves:", len(tree))
print("replies per move:", [len(branch) for branch in tree])

### Step 2 — One recursive evaluator for every node type

A single function walks the tree. At a **leaf** (a plain integer) it just returns the score. At an internal node it first evaluates every child, *flipping* whose turn it is on the way down.

Then it combines the children's values depending on the node:
- **maximizing** (our turn) → take the `max`, since we'll pick the best for us.
- **chance** (random opponent) → take the **average**, since each reply is equally likely.
- **minimizing** (adversarial opponent) → take the `min`, since a perfect opponent picks the worst for us.

The only difference between minimax and expectimax is that one line: `min` versus `average`.

In [ ]:
def evaluate(node, maximizing, chance):
    # Leaf node: the utility is just the number itself.
    if isinstance(node, int):
        return node

    # Internal node: evaluate each child with the turn flipped.
    child_values = [evaluate(child, not maximizing, chance) for child in node]

    if maximizing:
        # Our turn: pick the move that's best for us.
        return max(child_values)

    if chance:
        # Random opponent: average over their equally-likely replies.
        return sum(child_values) / len(child_values)

    # Adversarial opponent: they pick the worst outcome for us.
    return min(child_values)

### Step 3 — Compare worst-case vs random opponent

We run the *same* tree twice from the root, where it's our turn (`maximizing=True`). With `chance=False` the opponent minimizes — that's classic **minimax**. With `chance=True` the opponent acts randomly — that's **expectimax**.

The random opponent usually gives a higher value, because they won't reliably punish us with their worst reply.

In [ ]:
minimax_value = evaluate(tree, maximizing=True, chance=False)   # adversarial opponent
expectimax_value = evaluate(tree, maximizing=True, chance=True)  # random opponent

print("minimax value    (worst-case opponent):", minimax_value)
print("expectimax value (random opponent)    :", expectimax_value)

## Visualize the data & results

_For Pac-Man choosing a move, does it matter whether the ghost chases optimally or wanders randomly?_

We'll score each Pac-Man move under both ghost models and chart them side by side.

### Step 1 — Model Pac-Man's moves and the ghost's replies

Each Pac-Man move (`left`, `stay`, `right`) leads to three possible ghost responses, each ending in a pellet score. A **worst-case** (adversarial) ghost forces the `min` of each move's outcomes; a **random** ghost gives the average instead.

In [ ]:
# Each Pac-Man move -> the ghost's 3 possible responses -> pellet scores.
tree = {
    "left":  [3, 12, 8],
    "stay":  [4, 6, 2],
    "right": [-1, 9, 14],
}

# Score each move under each ghost model.
minimax_scores = {move: min(scores) for move, scores in tree.items()}
expecti_scores = {move: sum(scores) / len(scores) for move, scores in tree.items()}

print("worst-case ghost:", minimax_scores)
print("random ghost    :", {m: round(v, 2) for m, v in expecti_scores.items()})

### Step 2 — Chart the two ghost models side by side

Red bars assume the worst-case ghost; green bars assume the random ghost. Notice how the *best* move can differ between the two models — that's the whole point of expectimax: modeling the opponent as random can change which move you'd choose.

In [ ]:
moves = ["left", "stay", "right"]

# Build the bar labels, heights, and colours for both ghost models.
labels = [m + " (worst)" for m in moves] + [m + " (random)" for m in moves]
values = [minimax_scores[m] for m in moves] + [round(expecti_scores[m], 2) for m in moves]
colors = ["#ff7b72"] * 3 + ["#7ee787"] * 3

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(labels, values, color=colors)

# Label each bar with its score.
for i, v in enumerate(values):
    ax.text(i, v, str(v), ha="center", va="bottom")

ax.set_title("Pac-Man move score by ghost model (worst-case vs random)")
plt.tight_layout()
plt.show()